# GIL · Ejercicio 1 — Elegir la herramienta de concurrencia correcta

**Consigna**
1. Dado un problema, clasificarlo: CPU-bound, I/O-bound o mixto.
2. Elegir entre `threading`, `asyncio`, `multiprocessing` o extensión nativa.
3. Justificar la elección.
4. Implementar una versión mínima y medir.
5. Discutir costos ocultos: serialización, memoria, cancelación, complejidad.
6. **Pregunta final:** ¿qué ocurriría si reemplazo `ProcessPoolExecutor` por `ThreadPoolExecutor` en un cálculo pesado?

Trabajamos con **tres problemas** (uno CPU, uno I/O, uno mixto) para ver las tres respuestas posibles.

Hay que ejecutar las celdas **en orden**. Los tiempos dependen de la máquina; lo importante es la *proporción* entre variantes.

In [ ]:
import sys, os, time, threading, asyncio, pickle
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import pandas as pd

sys.path.insert(0, os.getcwd())   # para poder importar el módulo que escribimos abajo

try:
    nucleos = len(os.sched_getaffinity(0))
except AttributeError:
    nucleos = os.cpu_count()

print("Python:        ", sys.version.split()[0])
print("Núcleos útiles:", nucleos)
print("GIL activo:    ", getattr(sys, "_is_gil_enabled", lambda: True)())
print("Switch interval:", sys.getswitchinterval(), "s")
if nucleos < 2:
    print("\nCon 1 solo núcleo los procesos no pueden ir más rápido. En Colab normalmente hay 2.")

def medir(fn, repeticiones=3):
    """Devuelve el MEJOR tiempo (s) de varias corridas: reduce el ruido de la máquina."""
    mejores = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        fn()
        mejores.append(time.perf_counter() - t0)
    return min(mejores)

def correr_async(coro):
    """Ejecuta una corrutina en un thread aparte con su propio event loop.
    Colab/Jupyter ya tienen un loop corriendo, por eso no usamos asyncio.run() directo."""
    res = {}
    def _t():
        res["v"] = asyncio.run(coro)
    t = threading.Thread(target=_t); t.start(); t.join()
    return res["v"]

Python:         3.13.18
Núcleos útiles: 2
GIL activo:     True
Switch interval: 0.005 s


In [ ]:
%%writefile tareas_a.py
import time, os

def cpu_work(n):
    """Trabajo de CPU puro: un loop de Python."""
    total = 0
    for i in range(n):
        total += i * i
    return total

def contar_primos(rango):
    """Problema 1 (CPU): cuenta primos en [a, b) por división de prueba."""
    a, b = rango
    cuenta = 0
    for n in range(max(a, 2), b):
        i = 2
        es_primo = True
        while i * i <= n:
            if n % i == 0:
                es_primo = False
                break
            i += 1
        if es_primo:
            cuenta += 1
    return cuenta

def descargar(i):
    """Problema 2 (I/O): una 'request' que solo espera la red."""
    time.sleep(0.25)
    return 200

def fase_io(i):
    time.sleep(0.2)          # espera de red/disco

def fase_cpu(i):
    return cpu_work(2_000_000)   # procesamiento del contenido

def item_completo(i):
    """Problema 3 (mixto): primero espera (I/O) y después procesa (CPU)."""
    fase_io(i)
    return fase_cpu(i)

def largo_de(datos):
    return len(datos)

def info_proceso(_):
    rss = 0
    with open("/proc/self/status") as f:
        for linea in f:
            if linea.startswith("VmRSS"):
                rss = int(linea.split()[1]) / 1024   # kB -> MB
    return os.getpid(), round(rss, 1)

def tarea_larga(i):
    time.sleep(1)
    return i

Writing tareas_a.py


In [ ]:
import tareas_a as T
RESULTADOS = []   # acá juntamos todo para la tabla final

## Paso 1 · Clasificar: ¿en qué se va el tiempo?

- CPU ≈ reloj → el programa casi no espera, **calcula** → *CPU-bound*
- CPU ≪ reloj → pasa casi todo el tiempo esperando → *I/O-bound*
- Intermedio → *mixto*

In [ ]:
def perfil(fn):
    r0, c0 = time.perf_counter(), time.process_time()
    fn()
    reloj, cpu = time.perf_counter() - r0, time.process_time() - c0
    ratio = cpu / reloj
    if ratio > 0.85:   tipo = "CPU-bound"
    elif ratio < 0.15: tipo = "I/O-bound"
    else:              tipo = "Mixto"
    return reloj, cpu, ratio, tipo

# Problema 1: contar primos (8 tramos del mismo tamaño)
TRAMOS = [(i * 20_000, (i + 1) * 20_000) for i in range(8)]
def p1_secuencial(): return [T.contar_primos(t) for t in TRAMOS]

# Problema 2: 16 "descargas"
def p2_secuencial(): return [T.descargar(i) for i in range(16)]

# Problema 3: 8 ítems que esperan y luego procesan
def p3_secuencial(): return [T.item_completo(i) for i in range(8)]

filas = []
for nombre, fn in [("P1 · contar primos", p1_secuencial),
                   ("P2 · 16 descargas", p2_secuencial),
                   ("P3 · esperar + procesar", p3_secuencial)]:
    reloj, cpu, ratio, tipo = perfil(fn)
    filas.append({"Problema": nombre, "Reloj (s)": round(reloj, 2), "CPU (s)": round(cpu, 2),
                  "CPU/Reloj": round(ratio, 2), "Clasificación": tipo})
display(pd.DataFrame(filas))

,Problema,Reloj (s),CPU (s),CPU/Reloj,Clasificación
0,P1 · contar primos,0.52,0.52,1.00,CPU-bound
1,P2 · 16 descargas,4.00,0.00,0.00,I/O-bound
2,P3 · esperar + procesar,3.13,1.53,0.49,Mixto


### Conclusión del paso 1
Antes de elegir herramienta hay que saber **qué está haciendo el programa mientras tarda**. Si el procesador está ocupado todo el tiempo, el problema es de cálculo; si casi no trabaja y solo espera, es de entrada/salida; y si hace ambas cosas, es mixto. Medir esto lleva un minuto y evita elegir la herramienta por intuición.

## Paso 2 · Elegir y justificar

**P1 — Contar primos (CPU-bound) → `multiprocessing` (`ProcessPoolExecutor`).**
1. El tiempo se va en ejecutar bytecode de Python, y el GIL deja que un solo thread lo haga a la vez, así que los threads no suman velocidad.
2. Cada proceso tiene su propio intérprete y su propio GIL, por lo que puede ocupar un núcleo distinto en paralelo real.
3. Los datos de entrada son chicos (dos enteros) y el resultado también, así que el costo de serializar es despreciable frente al cálculo.

**P2 — 16 descargas (I/O-bound) → `asyncio` (o `threading` si la librería es bloqueante).**
1. El programa no calcula nada: solo espera respuestas, y mientras espera no necesita CPU.
2. Un thread (o una corrutina) que espera suelta el GIL, así que miles de esperas pueden solaparse con un único núcleo.
3. `asyncio` lo hace con muy poca memoria y sin locks; `threading` es la alternativa cuando la librería de red no es asíncrona.

**P3 — Esperar y luego procesar (mixto) → híbrido: threads/asyncio para la espera + procesos para el cálculo.**
1. Tiene dos cuellos de botella distintos: la espera se arregla con concurrencia liviana y el cálculo necesita paralelismo real.
2. Con solo threads, la espera se solapa pero el cálculo queda serializado por el GIL; con solo procesos, se desperdician procesos esperando.
3. Combinar ambos ataca cada fase con la herramienta adecuada, a cambio de más complejidad en el código.

**La extensión nativa (C/Rust/NumPy)** sería la opción para P1 si el loop de primos fuera el corazón de un sistema en producción: mover el loop a código compilado lo acelera por sí solo y además puede soltar el GIL. Para este ejercicio no hace falta, pero es la alternativa a considerar cuando "más procesos" ya no alcanza.

## Paso 3 · Implementar y medir
### P1 — CPU-bound: secuencial vs threads vs procesos

In [ ]:
pool_proc = ProcessPoolExecutor(max_workers=2)
list(pool_proc.map(T.cpu_work, [1, 1]))      # arranca los 2 procesos antes de medir

def p1_threads():
    with ThreadPoolExecutor(2) as ex: return list(ex.map(T.contar_primos, TRAMOS))
def p1_procesos():
    return list(pool_proc.map(T.contar_primos, TRAMOS))

assert p1_secuencial() == p1_procesos() == p1_threads()   # las 3 versiones dan lo mismo

t_seq = medir(p1_secuencial); t_thr = medir(p1_threads); t_prc = medir(p1_procesos)
for var, t in [("Secuencial", t_seq), ("2 threads", t_thr), ("2 procesos", t_prc)]:
    RESULTADOS.append(("P1 primos (CPU)", var, t))
display(pd.DataFrame({"Variante": ["Secuencial", "2 threads", "2 procesos"],
                      "Tiempo (s)": [round(t_seq,2), round(t_thr,2), round(t_prc,2)],
                      "Speedup": [1.0, round(t_seq/t_thr,2), round(t_seq/t_prc,2)]}))

,Variante,Tiempo (s),Speedup
0,Secuencial,0.51,1.00
1,2 threads,0.82,0.62
2,2 procesos,0.50,1.01


### P2 — I/O-bound: secuencial vs threads vs asyncio

In [ ]:
async def _descargar_async(i):
    await asyncio.sleep(0.25)
    return 200
async def _todas_async():
    return await asyncio.gather(*[_descargar_async(i) for i in range(16)])

def p2_threads():
    with ThreadPoolExecutor(16) as ex: return list(ex.map(T.descargar, range(16)))
def p2_asyncio():
    return correr_async(_todas_async())

t_seq = medir(p2_secuencial, 1); t_thr = medir(p2_threads); t_asy = medir(p2_asyncio)
for var, t in [("Secuencial", t_seq), ("16 threads", t_thr), ("asyncio", t_asy)]:
    RESULTADOS.append(("P2 descargas (I/O)", var, t))
display(pd.DataFrame({"Variante": ["Secuencial", "16 threads", "asyncio"],
                      "Tiempo (s)": [round(t_seq,2), round(t_thr,2), round(t_asy,2)],
                      "Speedup": [1.0, round(t_seq/t_thr,2), round(t_seq/t_asy,2)]}))

,Variante,Tiempo (s),Speedup
0,Secuencial,4.00,1.00
1,16 threads,0.25,15.79
2,asyncio,0.25,15.91


### P3 — Mixto: secuencial vs threads vs procesos vs híbrido

In [ ]:
ITEMS = list(range(8))

def p3_threads():
    with ThreadPoolExecutor(8) as ex: return list(ex.map(T.item_completo, ITEMS))
def p3_procesos():
    return list(pool_proc.map(T.item_completo, ITEMS))
def p3_hibrido():
    # threads para la espera; cuando termina la espera, el cálculo se manda a los procesos
    def tarea(i):
        T.fase_io(i)
        return pool_proc.submit(T.fase_cpu, i).result()
    with ThreadPoolExecutor(8) as ex: return list(ex.map(tarea, ITEMS))

assert p3_secuencial() == p3_threads() == p3_procesos() == p3_hibrido()

t_seq = medir(p3_secuencial, 2); t_thr = medir(p3_threads, 2)
t_prc = medir(p3_procesos, 2);   t_hib = medir(p3_hibrido, 2)
nombres = ["Secuencial", "8 threads", "2 procesos", "Híbrido (threads + procesos)"]
tiempos = [t_seq, t_thr, t_prc, t_hib]
for v, t in zip(nombres, tiempos): RESULTADOS.append(("P3 mixto", v, t))
display(pd.DataFrame({"Variante": nombres, "Tiempo (s)": [round(t,2) for t in tiempos],
                      "Speedup": [round(t_seq/t,2) for t in tiempos]}))

,Variante,Tiempo (s),Speedup
0,Secuencial,3.19,1.00
1,8 threads,2.36,1.35
2,2 procesos,2.54,1.26
3,Híbrido (threads + procesos),1.95,1.64


### Conclusión del paso 3
A raíz de los resultados obtenidos se concluye en las siguientes cosas: en el problema de **cálculo**, los threads no ayudaron (el GIL los hace turnarse) y los procesos sí, siempre que haya más de un núcleo disponible. En el problema de **espera**, threads y `asyncio` lograron lo mismo: terminar en el tiempo de *una* espera en vez de la suma de todas. Y en el **mixto**, ninguna herramienta sola fue la mejor: el enfoque híbrido combina lo bueno de cada una. No existe "la" herramienta correcta, sino la que ataca el cuello de botella real del problema.

## Paso 4 · Costos ocultos

### 4.1 Serialización
Los threads comparten memoria: pasarles datos es pasar una referencia. Los procesos **no**: todo argumento y todo resultado se serializa con `pickle`, se copia a través de un pipe y se reconstruye del otro lado.

In [ ]:
datos = list(range(3_000_000))
print(f"Tamaño serializado: {len(pickle.dumps(datos)) / 1e6:.1f} MB")

with ThreadPoolExecutor(1) as tp:
    tp.submit(T.largo_de, [1]).result()                     # calentamiento
    t_thread = medir(lambda: tp.submit(T.largo_de, datos).result())
t_proc = medir(lambda: pool_proc.submit(T.largo_de, datos).result())

print(f"Pasar la lista a un THREAD:  {t_thread*1000:8.2f} ms")
print(f"Pasar la lista a un PROCESO: {t_proc*1000:8.2f} ms   (≈ {t_proc/max(t_thread,1e-9):,.0f}× más)")
print("La función solo hace len(): todo ese tiempo extra es puro transporte de datos.")

Tamaño serializado: 14.9 MB
Pasar la lista a un THREAD:      0.08 ms
Pasar la lista a un PROCESO:   255.11 ms   (≈ 3,367× más)
La función solo hace len(): todo ese tiempo extra es puro transporte de datos.


### 4.2 Memoria
Cada proceso es un intérprete de Python completo. Miramos cuánta memoria residente (RSS) usa cada worker comparado con el proceso principal.

In [ ]:
principal_pid, principal_rss = T.info_proceso(None)
workers = sorted(set(pool_proc.map(T.info_proceso, range(8))))
print(f"Proceso principal (pid {principal_pid}): {principal_rss} MB")
for pid, rss in workers:
    print(f"Worker           (pid {pid}): {rss} MB")
print(f"Total aprox. con {len(workers)} workers: {principal_rss + sum(r for _, r in workers):.0f} MB")
print("Los threads, en cambio, comparten la memoria del proceso principal (solo suman su stack).")
print("(En Linux con 'fork' parte de esa memoria se comparte copy-on-write, así que el RSS exagera un poco.)")

Proceso principal (pid 472): 301.6 MB
Worker           (pid 1037): 115.4 MB
Worker           (pid 1037): 115.7 MB
Worker           (pid 1038): 112.4 MB
Worker           (pid 1038): 112.7 MB
Total aprox. con 4 workers: 758 MB
Los threads, en cambio, comparten la memoria del proceso principal (solo suman su stack).
(En Linux con 'fork' parte de esa memoria se comparte copy-on-write, así que el RSS exagera un poco.)


### 4.3 Cancelación

In [ ]:
# Threads: cancelamos 6 tareas de 1 s con solo 2 workers
ex = ThreadPoolExecutor(2)
futs = [ex.submit(T.tarea_larga, i) for i in range(6)]
time.sleep(0.2)
resultado_cancel = [f.cancel() for f in futs]
t0 = time.perf_counter()
ex.shutdown(wait=True)
print("¿Se pudo cancelar cada una?:", resultado_cancel)
print(f"→ las que ya estaban corriendo NO se cancelan; esperar el cierre tardó {time.perf_counter()-t0:.2f} s")

# asyncio: cancelación real en el próximo 'await'
async def demo_cancel():
    tarea = asyncio.create_task(asyncio.sleep(10))
    await asyncio.sleep(0.1)
    tarea.cancel()
    try:
        await tarea
    except asyncio.CancelledError:
        return "cancelada"
t0 = time.perf_counter()
print(f"asyncio: tarea de 10 s → {correr_async(demo_cancel())} en {time.perf_counter()-t0:.2f} s")

¿Se pudo cancelar cada una?: [False, False, True, True, True, True]
→ las que ya estaban corriendo NO se cancelan; esperar el cierre tardó 0.80 s
asyncio: tarea de 10 s → cancelada en 0.10 s


Lo mismo ocurre con los procesos: una tarea que ya empezó no se puede cancelar limpiamente; la única salida es matar el proceso, y en un `ProcessPoolExecutor` eso rompe el pool entero (`BrokenProcessPool`). `asyncio` cancela bien, pero **solo en los puntos `await`**: un loop de cálculo sin `await` no se puede interrumpir.

### 4.4 Complejidad

In [ ]:
# Con procesos, la función debe poder serializarse: una lambda NO puede.
with ProcessPoolExecutor(1) as p:
    try:
        p.submit(lambda: 1).result()
    except Exception as e:
        print("Error con lambda en ProcessPool →", type(e).__name__)
        print("   ", str(e)[:110])

# Con threads funciona sin problema:
with ThreadPoolExecutor(1) as p:
    print("Misma lambda en ThreadPool →", p.submit(lambda: 1).result())

Error con lambda en ProcessPool → PicklingError
    Can't pickle <function <lambda> at 0x7f808d38c720>: attribute lookup <lambda> on __main__ failed
Misma lambda en ThreadPool → 1


| Herramienta | Serialización | Memoria | Cancelación | Complejidad |
|---|---|---|---|---|
| `threading` | No (memoria compartida) | Baja | No se puede matar un thread | Locks, condiciones de carrera |
| `asyncio` | No | Muy baja | Buena, en cada `await` | Todo el código debe ser `async` |
| `multiprocessing` | **Sí, pickle** (costosa con datos grandes) | **Alta** (un intérprete por proceso) | Difícil (matar rompe el pool) | Funciones/argumentos "picklables", arranque de procesos |
| Extensión nativa | Casi nula (buffers) | Baja | Depende | Compilar, distribuir, mantener otro lenguaje |

###Conclusión del paso 4
Los procesos dan paralelismo real, en caso de disponer los núcleos para realizarlo, pero no son gratis: hay que **copiar los datos** en cada ida y vuelta, cada proceso **consume memoria** de un intérprete entero, cancelar es complicado y el código tiene más restricciones (por ejemplo, nada de lambdas). Por eso conviene mandar a los procesos **tareas grandes con datos chicos**, no muchas tareas diminutas con datos grandes. Y si el problema es de espera, usar procesos es pagar todos esos costos para nada.

## Paso 5 · Pregunta final
### ¿Qué ocurriría si reemplazo `ProcessPoolExecutor` por `ThreadPoolExecutor` en un cálculo pesado?


In [ ]:
TAREAS = [3_000_000] * 4
def calc_secuencial(): return [T.cpu_work(n) for n in TAREAS]
def calc_threads_2():
    with ThreadPoolExecutor(2) as ex: return list(ex.map(T.cpu_work, TAREAS))
def calc_threads_4():
    with ThreadPoolExecutor(4) as ex: return list(ex.map(T.cpu_work, TAREAS))
def calc_procesos_2():
    return list(pool_proc.map(T.cpu_work, TAREAS))

ts = [medir(calc_secuencial), medir(calc_threads_2), medir(calc_threads_4), medir(calc_procesos_2)]
nombres = ["Secuencial", "ThreadPool (2)", "ThreadPool (4)", "ProcessPool (2)"]
for v, t in zip(nombres, ts): RESULTADOS.append(("Pregunta final: cálculo pesado", v, t))
display(pd.DataFrame({"Variante": nombres, "Tiempo (s)": [round(t,2) for t in ts],
                      "Speedup": [round(ts[0]/t,2) for t in ts]}))
pool_proc.shutdown()

,Variante,Tiempo (s),Speedup
0,Secuencial,1.13,1.00
1,ThreadPool (2),1.19,0.95
2,ThreadPool (4),1.17,0.97
3,ProcessPool (2),1.27,0.89


### Respuesta
**Se pierde el paralelismo y el cálculo no se acelera.** Con `ThreadPoolExecutor`, todos los threads comparten un único GIL, así que solo uno ejecuta bytecode de Python en cada instante. Los threads se turnan, y el resultado es un tiempo igual al secuencial o incluso un poco peor, por el costo de cambiar de thread (y, mirando la fila de 4 threads, agregar más threads no arregla nada). Con `ProcessPoolExecutor`, cada worker tiene su propio GIL y puede ocupar un núcleo distinto.

La excepción: si la tarea "pesada" pasa su tiempo dentro de código nativo que **suelta el GIL** (NumPy con arrays grandes, por ejemplo), los threads sí pueden paralelizar. Pero para cálculo en Python puro, ProcessPool → ThreadPool es un error de rendimiento silencioso: el programa sigue funcionando y da el mismo resultado, solo que no más rápido.

In [ ]:
resumen = pd.DataFrame(RESULTADOS, columns=["Experimento", "Variante", "Tiempo (s)"])
resumen["Speedup vs primera variante"] = resumen.groupby("Experimento")["Tiempo (s)"].transform(lambda s: s.iloc[0] / s).round(2)
resumen["Tiempo (s)"] = resumen["Tiempo (s)"].round(2)
display(resumen)

,Experimento,Variante,Tiempo (s),Speedup vs primera variante
0,P1 primos (CPU),Secuencial,0.53,1.00
1,P1 primos (CPU),2 threads,0.54,0.97
2,P1 primos (CPU),2 procesos,0.86,0.61
3,P1 primos (CPU),Secuencial,0.51,1.04
4,P1 primos (CPU),2 threads,0.82,0.65
5,P1 primos (CPU),2 procesos,0.50,1.05
6,P2 descargas (I/O),Secuencial,4.00,1.00
7,P2 descargas (I/O),16 threads,0.25,15.79
8,P2 descargas (I/O),asyncio,0.25,15.91
9,P3 mixto,Secuencial,3.19,1.00


## Conclusión final del ejercicio
**Antes de elegir una herramienta de concurrencia hay que preguntarse qué está esperando el programa**. Si espera a la red o al disco, alcanza con `asyncio` o threads, que son baratos. Si está calculando, hacen falta procesos (o código nativo) para usar más de un núcleo, porque el GIL impide que dos threads ejecuten Python al mismo tiempo. Si hace las dos cosas, se combinan.

El paralelismo con procesos tiene un precio: copiar datos, gastar memoria, complicar la cancelación y restringir cómo se escribe el código. Por eso no es "siempre mejor": es mejor cuando la tarea es lo bastante grande como para compensar esos costos.

Es ideal **medir primero**. La intuición sobre qué es lento suele fallar, y un `time.perf_counter()` ahorra muchas horas de optimizar lo que no era.